# Analysis preamble
This notebook is supposed to create the analysis tools and functions for a specific experiment. It includes most of the specific names and used in part general routines/frameworks/libraries. As a result this notebook or derived copies contain most of the relevant experiment-specific code and potentially hard that and can be adjusted without much crosstalk. 

## General imports
All imports needed should be placed here, the section typically also includes imports of tools and setup of the notebooks in order to make it simple to create new notebooks and 

In [ ]:
%matplotlib widget
%pylab
# import sys
# sys.path.insert(0,'./escape-fel')
from escape import swissfel
from escape import utilities
from escape.utilities import nfigure, nsubplots, plot2D, cell2function
from escape.swissfel import detector
import escape.storage
from escape.swissfel import timetool
import warnings
import pathlib
import os
import traceback
warnings.filterwarnings("ignore", category=DeprecationWarning) 
from pathlib import Path

%load_ext autoreload
%autoreload 2


## Experiment specific values
The following cell includes things like paths and calibration constants to some cenral location. those should ideally be valid over the entire course of one experiment, as changing those can cause unintended issues. 
### Names, paths, etc

In [ ]:
exp_name = '23j_ovuka'
results_directory = '/sf/bernina/exp/example_data/analysis_notebooks/reduced_data_zarr/'

### Calibration constants

In [3]:
tt_calib = array([-4.29795532e-19, -2.33229832e-15,  1.60788089e-12]) # ovuka run 1

## Region of interest definition
Example for changing regions of interest configured on per-run basis. 

In [4]:
def set_table(run):
    set_dict = {'roi_i0': (148,882,271,495),
                'roi_fluo': (1, 880,1,508),
                'roi_diff': '',
                'filt_i0': '',
                'filt_i0d': '',
                'filt_tt_pos': '',
                'filt_tt_ampl': ''}
                
    if run >-1 and run<=5:
        #set_dict['roi_i0'] = 
        set_dict['roi_fluo'] = (400, 700,100,400)
        #set_dict['roi_diff'] = (270,500,260,760)
        set_dict['roi_diff'] = (690, 820, 655 ,780)
        set_dict['filt_i0'] = (-5000, 2000)
    if run ==6:
        #set_dict['roi_i0'] = 
        set_dict['roi_fluo'] = (400, 700,100,400)
        #set_dict['roi_diff'] = (270,500,260,760)
        set_dict['roi_diff'] = (400, 900, 300 ,700)
        set_dict['filt_i0'] = (-5000, 2000)      
    if run ==7:
        set_dict['roi_fluo'] = (400, 700,100,400) #(333)peak
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (610, 670, 688 ,748)
        set_dict['filt_i0'] = (-5000, 2000)
    if run ==8:
        set_dict['roi_fluo'] = (400, 700,100,400) #(222)peak
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (585, 670, 666 ,760)
        set_dict['filt_i0'] = (-5000, 2000)
    if run >8 and run <= 29:
        set_dict['roi_fluo'] = (400, 700,100,400) #go back to (333)peak
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (610, 670, 688 ,748)
        set_dict['filt_i0'] = (-5000, 2000)       
    if run >29 and run<52:
        set_dict['roi_fluo'] = (400, 700,100,400) #(222)peak
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (585, 670, 666 ,760)
        set_dict['filt_i0'] = (-5000, 2000)   
    if run>=52 and run<69:
        #set_dict['roi_fluo'] =  #(222)peak
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (585, 670, 666 ,760)
        set_dict['filt_i0'] = (-5000, 2000)   
    if run>=69 and run<88:
        #set_dict['roi_fluo'] =  #(222)peak
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (584, 658, 665 ,769)
        set_dict['filt_i0'] = (-5000, 2000)  
    if run>=88 and run<134:
        set_dict['roi_fluo'] =  (643, 762,18,361) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        # set_dict['roi_diff'] = (584, 658, 665 ,769)
        set_dict['filt_i0'] = (-5000, 2000)  
    if run>=134 and run<140:
        #set_dict['roi_fluo'] =  (643, 762,18,361) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000)  
    if run == 134:
        set_dict['roi_fluo'] =  (536, 751, 325, 498) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        #set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000) 
    if run>=140 and run<143:
        set_dict['roi_fluo'] =  (538, 754, 26, 237) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        #set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000)  
        
    if run>=143 and run<146:
        set_dict['roi_fluo'] =  (538, 754, 26, 237) 
        set_dict['roi_diff'] = (613, 647, 700, 735)
        #set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000) 
        
    if run >= 147 and run < 153:
        set_dict['roi_fluo'] =  (536, 751, 325, 498) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        #set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000)     
    
    if run >= 153 and run < 221:
        set_dict['roi_fluo'] =  (425, 880, 115, 485) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        #set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000)     
    if run>=221 and run<239:
        #set_dict['roi_fluo'] =  (643, 762,18,361) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000)
    
    if run >= 239 and run < 286:
        set_dict['roi_fluo'] =  (425, 880, 115, 485) 
        #set_dict['roi_diff'] = (560, 680, 645 ,795)
        #set_dict['roi_diff'] = (618, 657, 696, 738)
        set_dict['filt_i0'] = (-5000, 2000) 
    if run >= 286 and run < 298:
        # set_dict['roi_fluo'] =  (425, 880, 115, 485) 
        # set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (600, 634, 744, 769)
        set_dict['filt_i0'] = (-5000, 2000) 
    if run >= 298 and run < 323:
        set_dict['roi_fluo'] =  (268, 971, 81, 417) 
        # set_dict['roi_diff'] = (560, 680, 645 ,795)
        # set_dict['roi_diff'] = (600, 634, 744, 769)
        set_dict['filt_i0'] = (-5000, 2000) 
    if run >= 323 and run < 325:
        # set_dict['roi_fluo'] =  (425, 880, 115, 485) 
        # set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (600, 634, 744, 769)
        set_dict['filt_i0'] = (-5000, 2000) 
    if run >= 325 and run < 371:
        # set_dict['roi_fluo'] =  (425, 880, 115, 485) 
        # set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (290, 612, 800, 1063)
        set_dict['filt_i0'] = (-5000, 2000) 
        
    if run >= 371 and run<666:
               # set_dict['roi_fluo'] =  (425, 880, 115, 485) 
        # set_dict['roi_diff'] = (560, 680, 645 ,795)
        set_dict['roi_diff'] = (260, 510, 795, 1075)
        set_dict['filt_i0'] = (-5000, 2000)  
    return set_dict

def ROIS(run):
    
    set_dict = set_table(run)
    ROIS_dict = {}

    # roi for fluorescence detector xrd.det_fluo
    ROIS_dict['bernina.xrd.det_fluo_keV'] ={'hor_region': set_dict['roi_fluo']} # to be adjusted!

    # roi for i0 detector det_i0
    ROIS_dict['bernina.det_i0_keV'] = {'scatter': set_dict['roi_i0']} # to be adjusted!

    # roi for diffraction detector xrd.det_diff
    ROIS_dict['bernina.xrd.det_diff_keV'] = {'peak': set_dict['roi_diff']} # to be adjusted!
    
    return ROIS_dict

## General raw data loading function 
This includes general data reduction. NB: Changing regions of interest need to be dealt with carefully. Global variables used above might not be a good choice for some signal types. 
As example here the "peak" roi is kept more variable. 

In [5]:
def load_dset(
    runno,
    clear_result_file=False,
    result_type='zarr',
    result_filename='auto',
    results_directory= results_directory,
    roi_peak='global', # either good roi {'peak': (270,500,260,760)} or 'global' (graphically select roi using get_roi function)
):
    if result_filename == 'auto':
        result_filename = f'run{runno:04d}_reduced'
        
    d = swissfel.load_dataset_from_scan(
        exp_name=exp_name,
        run_numbers=[runno], 
        # checknstore_parsing_result='./',
        result_type = result_type,
        result_filename = result_filename,
        results_directory = results_directory,
        clear_result_file = clear_result_file,
    )
    
    
    # flag for Pump "off-like" reference used for both pump/probe and timetool signal sorting.
    try:
        d.append(d.data_raw['SAR-CVME-TIFALL5:EvtSet'][:,25],name='bernina.pump_delayed')
    except:
        print('WARNING: no event data set found!')
    
    # timetool analysis
    try:
        # time tool calculation of edge position.
        d.append(timetool.get_relative_profiles(
            d.bernina.tt_kb.spectrum_signal,
            d.bernina.pump_delayed,
            fac_references=1, Npids=200)[:,1500:500:-1],name='bernina.tt_kb.spec_rel')

        # timetool: convolution reference step is hardcoded here in total size and and width, might need to be changed.  
        ttref = timetool.get_reference_function(sigma_px=17,reflen=150)
        ttres = d.bernina.tt_kb.spectrum_signal.categorize(
            d.bernina.tt_kb.spec_rel.map_index_blocks(lambda d: np.asarray([timetool.find_signal(td,ttref) for td in d]),new_element_size=[2],dtype=float))
        d.append(ttres[:,0],name='bernina.tt_kb.ttpos')
        d.append(ttres[:,1],name='bernina.tt_kb.ttamp')

        # applying timetool calibration
        d.append(d.bernina.tt_kb.ttpos.map_index_blocks(lambda data: np.polyval(tt_calib, data)),
                name='bernina.tt_kb.tt_time')
    except:
        print('WARNING: timetool analysis failed with trace -->')
        traceback.print_exc(chain=False)
        
        
    # detector to keV conversion
    
    for tdet in ['bernina.xrd.det_diff','bernina.xrd.det_fluo','bernina.det_i0']:
        try:
            fac = eval(f'd.status_run_start.{tdet}.config_daq.rounding_factor_keV')
            d.append(d.datasets[tdet]*fac,name=tdet+'_keV')
        except:
            print('WARNING: detector keV conversion failed with trace -->')
            traceback.print_exc(chain=False)
        
    # detector roi creation and summing
    for tdet in ['bernina.xrd.det_diff_keV','bernina.xrd.det_fluo_keV','bernina.det_i0_keV']:
        print(tdet)
        print(ROIS(runno)[tdet])
        try:
            if tdet == 'bernina.xrd.det_diff_keV':
                if not roi_peak=='global':
                    out = d.datasets[tdet].scan.tools.get_regions_of_interest_rectangular_2D(rois=roi_peak,show=False)
                else:
                    out = d.datasets[tdet].scan.tools.get_regions_of_interest_rectangular_2D(rois=ROIS(runno)[tdet],show=False)
            else:
                out = d.datasets[tdet].scan.tools.get_regions_of_interest_rectangular_2D(rois=ROIS(runno)[tdet],show=False)
            for rname,rarray in out.result.items():
                d.append(rarray.sum(axis=(1,2)),name='_'.join([tdet,rname,'sum']))
            print(tdet)
            print(ROIS(runno)[tdet])
        except:
            print(f'WARNING: roi calc failed for {tdet} with trace -->')
            traceback.print_exc(chain=False)

    return d


def get_rois(det,rois={}):
    """det should be a 2D data containing escape array (i.e. with events that'd 3D array
    E.g. cell: out = get_rois(det)\n
    next cell: rois = out.rois\n"""
    
    out = det.scan.tools.get_regions_of_interest_rectangular_2D(rois=rois)
    print("Select roi in GUI element below, name it, and then your roi definition will be part of your output.")
    return out
    

## Functions to reduce dataset and total workflow

### Reduce dataset
This example uses the load_data function which

In [6]:
# function to reduce dataset after load
def reduce_dset(d, close_result_file=True, max_element_size=5000, **kwargs):
    """reduce dataset using the store dataset_max element function, 
    i.e. all smaller data will be read, calculated if required and stored to a new esc file.
    close_results_file: Closing the results file after reduction, safer, but the dataset becomes non-usable in the current session."""
    d.store_datasets_max_element_size(max_element_size=max_element_size, **kwargs)
    if close_result_file:
        try:
            d.results_file.close()
        except:
            pass
    return d

## Functions to load and plot reduced data

In [7]:
def load_reduced_data(runno):
    file = Path(results_directory)/Path(f'run{runno:04d}_reduced.esc.zarr')
    return escape.DataSet.load_from_result_file(file)


# to be adjusted to names used above for I/I0 etc...
def ana_timescan(d,tbinsize = 10e-15):
    
    tmp_on = (d.bernina.xrd.det_diff/d.bernina.jet.monint)[~d.bernina.pump_delayed].compute()
    tmp_off = (d.bernina.jet.totem/d.bernina.jet.monint)[d.bernina.pump_delayed].compute()
    resort = d.bernina.jet.totem.get_index_array(N_index_aggregation=1000)
    res_on = resort.categorize(tmp_on)
    res_off = resort.categorize(tmp_off)

    pprat = res_on.scan/res_off.scan.mean(axis=0)

    t = tmp_on.scan.par_steps.iloc[:,0]
    tt = d.bernina.tt_kb.tt_time.compute()
    tt_med = tt.nanmedian()
    t_tt = tt.scan+t
    t_tt_binned = t_tt.digitize(
        np.arange(np.nanmin(t)+tt_med,np.nanmax(t)+tt_med,tbinsize))
    pprat_tt = t_tt_binned.categorize(pprat)
    
    nfigure('timescan')
    pprat_tt.scan.plot(label='timetool corrected')
    tmp_on.categorize(pprat).scan.plot(label='scan raw')
    plt.legend()
    return pprat_tt

## Correlation analysis

In [11]:
from scipy.optimize import curve_fit

def Corr_Plot(runno, i0, sig, rng = ''):
    fig, ax = plt.subplots(figsize = (12, 8))
    fig.suptitle('Run {}, correlations'.format(runno), fontsize=16)
    ax.scatter(i0, sig, marker = 'x', alpha = 0.2, label = 'all')
    if isinstance(rng, str):
        pass
    else:
        filt = np.logical_and(i0>rng[0], i0<rng[1])
        ax.scatter(i0[filt], sig[filt], marker = 'x', alpha = 0.2, label = 'filtered')
    ax.set(title = 'correlation plot', xlabel = 'i0', ylabel = 'sig')
    ax.legend(loc = 'best')
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    
def SNR_Eval(runno, i0, sig, edges):
    
    edges = np.sort(edges)
    fkt = lambda x, a, b, c: a + b*x + c * x**2
    
    fig, ax = plt.subplots(ncols = 2, nrows = 2, figsize = (12, 8))
    fig.suptitle('Run {}, SNR eval'.format(runno), fontsize=16)
    ax[0,0].scatter(i0, sig, color = 'blue', marker = 'x', alpha = 0.2, label = 'all')
    ax[0,0].vlines(edges, np.nanmin(sig), np.nanmax(sig), color = 'orange', label = 'edges')
    
    filt = np.logical_and(i0>edges[0], i0<edges[-1])
    #par, cov = curve_fit(fkt, i0[filt], sig[filt], p0 = (0,1,0))
    par, cov = curve_fit(fkt, i0, sig, p0 = (0,1,0))
    
    N = len(edges) -1 
    i0_avg = np.zeros(N)
    sig_avg = np.zeros(N)
    sig_std = np.zeros(N)
    sig_avg_corr = np.zeros(N)
    sig_std_corr = np.zeros(N)
    for i in np.arange(N):
        filt = np.logical_and(i0>edges[i], i0<edges[i+1])
        i0_avg[i] = np.nanmean(i0[filt])
        sig_avg[i] = np.nanmean(sig[filt])
        sig_std[i] = np.nanstd(sig[filt])
        sig_avg_corr[i] = np.nanmean(sig[filt] - fkt(i0[filt], *par))
        sig_std_corr[i] = np.nanstd(sig[filt] - fkt(i0[filt], *par))
        if i == N//2:
            ax[1,0].hist(sig[filt] -  fkt(i0[filt], *par), bins = int(np.sqrt(np.sum(filt))))
            ax[1,0].set(title = 'hist sig deviation bin {:.4}'.format(i0_avg[i]), xlabel = 'val', ylabel = 'nr')
    
    ax[0,0].scatter(i0_avg, sig_avg, color = 'red', marker = 'x', label = 'avg')
    ax[0,0].plot(i0, fkt(i0, *par), label = 'fit')
    ax[0,0].set(title = 'correlation plot', xlabel = 'i0', ylabel = 'sig')
    ax[0,0].legend(loc = 'best')
    
    ax[0,1].scatter(i0_avg, sig_std_corr, marker = 'x')
    ax[0,1].set(title = 'std analysis', xlabel = 'i0', ylabel = 'std')
    ax[1,1].scatter(i0_avg, sig_avg_corr/sig_std_corr, marker = 'x')
    ax[1,1].set(title = 'SNR', xlabel = 'i0', ylabel = 'SNR')
                        
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    
    


##  Filters and motor positions

In [12]:
def Get_Filter(dr):
    key_dict = {'i0': dr.bernina.det_i0_keV_scatter_sum.data,
                'i0d': dr.bernina.mon_opt_new.intensity.data}
    try:
        key_dict['tt_ampl'] = dr.bernina.tt_kb.ttamp.data
        key_dict['tt_pos'] = dr.bernina.tt_kb.ttpos.data
        fig, ax = plt.subplots(ncols = 2, nrows = 2, figsize = (12, 8))
        ax = ax.ravel()
    except:    
        fig, ax = plt.subplots(ncols = 2, nrows = 1, figsize = (12, 8))
        ax = ax.ravel()
        print('no tt')
    
    runno = int(dr.results_file.filename[47:51])
    setting_dict = set_table(runno)
    
    
    N = len(dr.bernina.det_i0_keV_scatter_sum.data)
    las_on = np.logical_not(dr.bernina.pump_delayed.data)
    las_off = dr.bernina.pump_delayed.data
    filt = np.zeros(N)== 0
    filt_on = np.logical_not(dr.bernina.pump_delayed.data)
    filt_off = dr.bernina.pump_delayed.data
    
    fig.suptitle('Run {}, filters'.format(runno), fontsize=16)
    
    for ki, key in enumerate(key_dict):
        hist, edges = np.histogram(key_dict[key], bins = int(np.sqrt(N)))
        ax[ki].hist(key_dict[key], edges, label = 'all')
        ax[ki].hist(key_dict[key][las_on], edges, label = 'on')
        ax[ki].hist(key_dict[key][las_off], edges, label = 'off')
        
        ax[ki].set(title = 'filter {}'.format(key), xlabel = 'value', ylabel = 'nr')
        ax[ki].legend(loc = 'best')
        
        if isinstance(setting_dict['filt_{}'.format(key)] , str):
            print('filter for {} not defined'.format(key))
        elif 'tt' in key:
            filt_on  = np.logical_and(filt_on, np.logical_and(key_dict[key]>setting_dict['filt_{}'.format(key)][0], 
                                                              key_dict[key]<setting_dict['filt_{}'.format(key)][1]))
            filt[las_on] = np.logical_and(filt[las_on], np.logical_and(key_dict[key][las_on]>setting_dict['filt_{}'.format(key)][0], 
                                                                       key_dict[key][las_on]<setting_dict['filt_{}'.format(key)][1]))
            ax[ki].vlines(setting_dict['filt_{}'.format(key)], 0, np.max(hist), color = 'red')
        else:
            filt_on  = np.logical_and(filt_on, np.logical_and(key_dict[key]>setting_dict['filt_{}'.format(key)][0], 
                                                              key_dict[key]<setting_dict['filt_{}'.format(key)][1]))
            filt_off  = np.logical_and(filt_off, np.logical_and(key_dict[key]>setting_dict['filt_{}'.format(key)][0], 
                                                                key_dict[key]<setting_dict['filt_{}'.format(key)][1]))
            filt = np.logical_and(filt[las_on], np.logical_and(key_dict[key]>setting_dict['filt_{}'.format(key)][0], 
                                                               key_dict[key]<setting_dict['filt_{}'.format(key)][1]))
            ax[ki].vlines(setting_dict['filt_{}'.format(key)], 0, np.max(hist), color = 'red')
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    
    print('filtered ration all: {:.4}'.format(np.sum(filt)/N))
    print('filtered ratio on: {:.4}'.format(np.sum(filt_on)/np.sum(las_on)))
    print('filtered ratio off: {:.4}'.format(np.sum(filt_off)/np.sum(las_off)))
    return filt, filt_on, filt_off
                

def Get_Motor_Pos(dr):
    steps = dr.bernina.det_i0_keV_scatter_sum.scan.par_steps.iloc[:,0]
    pos_arr = np.concatenate([(np.zeros(dr.bernina.det_i0_keV_scatter_sum.scan[ii].data.shape[0])+1)*si for ii, si in enumerate(steps)])
    return pos_arr

## Plot Functions

User tools of Pvuka beamtime

In [13]:
from scipy.stats import binned_statistic

def Plot_Sig(runno, pos, i0, sig, edges, mode = 'avd', fit = '', FFT = False):
    
    bins = edges[:-1]+ 0.5*(edges[1]-edges[0])
    if FFT:
        fig, ax = plt.subplots(ncols = 2)
    else:
        fig, ax = plt.subplots()
        ax = [ax]
    fig.suptitle('Run {}, signal {}'.format(runno, mode), fontsize=16)
    
    if mode == 'avd':
        i0_sum, edges, index = binned_statistic(pos, i0, statistic = 'sum', bins = edges)
        sig_sum, edges, index = binned_statistic(pos, sig, statistic = 'sum', bins = edges)
        sig_sig = sig_sum/i0_sum
    elif mode == 'sts':
        sig_sig, edges, index = binned_statistic(pos, sig/i0, statistic = 'mean', bins = edges)
    ax[0].plot(bins, sig_sig, label = 'data')
    if isinstance(fit, str):
        pass
    else:
        par, cov = curve_fit(fit['fkt'], sig_sig, bins, p0 = fit['p0'])
        ax[0].plot(bins, fit['fkt'](bins, *par), label = 'fit')
        ax[0].legend(loc = 'best')
    
    if FFT:
        freq = np.fft.fftfreq(len(bins), bins[1]-bins[0])
        fft = np.fft.fft(sig_sig)
        ax[1].plot(freq[:len(bins)//2], abs(fft[:len(bins)//2])**2)
        
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    
    
def Plot_OnOff_Sig(runno, pos, i0, sig, on_bool, off_bool,mode = 'avd', fit = '', FFT = True):
    
    bins = edges[:-1]+ 0.5*(edges[1]-edges[0])
    if FFT:
        fig, ax = plt.subplots(ncols = 3)
    else:
        fig, ax = plt.subplots(ncols = 2)
    fig.suptitle('Run {}, on off signal, {}'.format(runno, mode), fontsize=16)
    
    if mode == 'avd':
        i0_on_sum, edges, index = binned_statistic(pos[on_bool], i0[on_bool], statistic = 'sum', bins = edges)
        i0_off_sum, edges, index = binned_statistic(pos[off_bool], i0[off_bool], statistic = 'sum', bins = edges)
        sig_on_sum, edges, index = binned_statistic(pos[on_bool], on[on_bool], statistic = 'sum', bins = edges)
        sig_off_sum, edges, index = binned_statistic(pos[off_bool], off[off_bool], statistic = 'sum', bins = edges)
        sig_on_corr = sig_on_sum/i0_on_sum
        sig_off_corr = sig_off_sum/i0_off_sum
        sig_sig = sig_on_corr/sig_off_corr-1
    elif mode == 'sts':
        sig_on_corr, edges, index = binned_statistic(pos[on_bool], sig[on_bool]/i0[on_bool], statistic = 'mean', bins = edges)
        sig_off_corr, edges, index = binned_statistic(pos[off_bool], sig[off_bool]/i0[off_bool], statistic = 'mean', bins = edges)
        sig_sig = sig_on_corr/sig_off_corr-1
        
    ax[0].plot(bins, sig_sig, label = 'data')
    print('slksdlksdlk')
    if isinstance(fit, str):
        pass
    else:
        par, cov = curve_fit(fit['fkt'], sig_sig, bins, p0 = fit['p0'])
        ax[0].plot(bins, fit['fkt'](bins, *par), label = 'fit')
        ax[0].legend(loc = 'best')
        print('fit par: {}'.format(par))
    
    if FFT:
        freq = np.fft.fftfreq(len(bins), bins[1]-bins[0])
        fft = np.fft.fft(sig_sig)
        ax[1].plot(freq[:len(bins)//2], abs(fft[:len(bins)//2])**2)
        
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    
    

In [1]:
from escape.storage import concatenate